In [4]:
import requests
import csv
import os
import re
import io
import calendar
from PyPDF2 import PdfReader

# desired location for all data, in this case a new 'data' folder:
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data')
# regex to match filenames like "01-20-2018.pdf" (case-insensitive).
# remove the .pdf extension before matching, so that part is optional in the regex.
DATE_PATTERN = re.compile(r"^(\d{2})-(\d{2})-(\d{4})$", re.IGNORECASE)

def download_pdfs(
        
    # url format for the PDFs, with a placeholder for the ID
    start_id=2274,
    end_id=49898,
    base_url="https://lawpd.com/DocumentCenter/View/{}",
    failure_csv="failures.csv"
):
    """
    Download PDFs by incrementing through IDs, skipping failures,
    organizing them by date if the filename is in MM-DD-YYYY format,
    or else placing them in a fallback 'no_date' folder with a
    best-effort headline-based filename. Logs errors to a CSV file.
    """

    # ensure the base data directory exists
    os.makedirs(DATA_DIR, exist_ok=True)

    # build a path to the failures.csv in the data directory
    failure_csv_path = os.path.join(DATA_DIR, failure_csv)

    total = end_id - start_id + 1

    # open CSV file to log failures
    with open(failure_csv_path, mode="w", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["ID", "Error"])  # header row

        for count, file_id in enumerate(range(start_id, end_id + 1), start=1):
            if count % 500 == 0:
                print(f"📦 Progress: {count} / {total} checked")

            url = base_url.format(file_id)

            try:
                response = requests.get(url, timeout=10)  # 10-second timeout
                status_code = response.status_code

                # check HTTP status
                if status_code != 200:
                    writer.writerow([file_id, f"HTTP status {status_code}"])
                    continue

                # check for PDF content
                content_type = response.headers.get("Content-Type", "").lower()
                if "pdf" not in content_type:
                    writer.writerow([file_id, f"Not a PDF (content-type: {content_type})"])
                    continue

                # attempt to get the raw server filename from Content-Disposition
                content_disp = response.headers.get("Content-Disposition", "")
                server_filename = get_filename_from_content_disposition(content_disp)

                # if the server doesn't provide a filename, fallback to ID #, e.g. "2274.pdf"
                if not server_filename:
                    server_filename = f"{file_id}.pdf"

                # ensure .pdf extension
                if not server_filename.lower().endswith(".pdf"):
                    server_filename += ".pdf"

                # try parsing a date from the filename
                year, month, day = parse_date_from_filename(server_filename)

                # if date is parseable, build the nested folder path
                if year is not None:
                    # example: 2018_law_pd_data -> 2018_january
                    year_folder = f"{year}_law_pd_data"
                    month_name = calendar.month_name[month].lower()  # 'january', 'february', etc.
                    month_folder = f"{year}_{month_name}"

                    year_folder_path = os.path.join(DATA_DIR, year_folder)
                    month_folder_path = os.path.join(year_folder_path, month_folder)

                    # create subdirectories
                    os.makedirs(month_folder_path, exist_ok=True)

                    # final path for the PDF
                    file_path = os.path.join(month_folder_path, server_filename)

                else:
                    # if we can't parse a date, try extracting the first page headline
                    pdf_bytes = response.content
                    headline = extract_pdf_headline(pdf_bytes)

                    if headline:
                        safe_headline = sanitize_filename(headline)
                        fallback_name = f"{file_id}_{safe_headline}.pdf"
                    else:
                        fallback_name = f"{file_id}_no_headline.pdf"

                    fallback_folder_path = os.path.join(DATA_DIR, "no_date")
                    os.makedirs(fallback_folder_path, exist_ok=True)

                    file_path = os.path.join(fallback_folder_path, fallback_name)

                # write the PDF content to disk
                with open(file_path, "wb") as out_file:
                    out_file.write(response.content)

                print(f"[{count}/{total}] ✅ Saved: {server_filename}")

            except Exception as e:
                # log any exception (network errors, parse errors, etc.)
                writer.writerow([file_id, str(e)])
                continue


def get_filename_from_content_disposition(content_disp):
    """
    Attempt to parse the filename= value from a Content-Disposition header.
    Returns None if not found.
    Example header: 'attachment; filename="01-20-2018.pdf"'
    """
    if "filename=" in content_disp.lower():
        parts = content_disp.split("filename=")
        if len(parts) > 1:
            # Remove surrounding quotes or semicolons
            filename_part = parts[1].strip().strip('"').strip(';')
            return filename_part
    return None


def parse_date_from_filename(filename):
    """
    Given a filename like '01-20-2018.pdf', parse it as MM-DD-YYYY.
    Returns (year, month, day) if successful, or (None, None, None) if not.
    """
    # strip off '.pdf'
    base_name = os.path.splitext(filename)[0]
    match = DATE_PATTERN.match(base_name)
    if not match:
        return (None, None, None)

    mm = int(match.group(1))
    dd = int(match.group(2))
    yyyy = int(match.group(3))

    # very basic date validity check
    if 1 <= mm <= 12 and 1 <= dd <= 31:
        return (yyyy, mm, dd)
    else:
        return (None, None, None)


def extract_pdf_headline(pdf_bytes):
    """
    Try reading the first page of a PDF to get the first line of text.
    Returns that line (string) or None if it fails or no text is found.
    """
    try:
        pdf_stream = io.BytesIO(pdf_bytes)
        pdf_reader = PdfReader(pdf_stream)
        if len(pdf_reader.pages) > 0:
            first_page = pdf_reader.pages[0]
            text = first_page.extract_text() or ""
            lines = text.splitlines()
            if lines:
                # return the first non-empty line
                return lines[0].strip()
        return None
    except:
        return None


def sanitize_filename(name):
    """
    remove or replace characters that are problematic in filenames,
    returning a safer string. Also truncates to a reasonable length.
    """
    # replace anything not alphanumeric, underscore, or dash with underscore
    safe = re.sub(r"[^a-zA-Z0-9_\-]+", "_", name)
    # limit length (here to 50 chars)
    return safe[:50]

In [ ]:
download_pdfs()

[1/47625] ✅ Saved: 01-11-2018.pdf
[2/47625] ✅ Saved: 01-12-2018.pdf
[3/47625] ✅ Saved: 01-13-2018.pdf
[4/47625] ✅ Saved: 01-14-2014.pdf
[5/47625] ✅ Saved: 01-15-2018.pdf
[6/47625] ✅ Saved: 01-16-2018.pdf
[7/47625] ✅ Saved: 01-17-2018.pdf
[8/47625] ✅ Saved: 01-18-2018.pdf
[9/47625] ✅ Saved: 01-19-2018.pdf
[10/47625] ✅ Saved: 01-20-2018.pdf
[11/47625] ✅ Saved: 01-21-2018.pdf
[12/47625] ✅ Saved: 01-22-2018.pdf
[13/47625] ✅ Saved: 01-23-2018.pdf
[14/47625] ✅ Saved: 01-24-2018.pdf
[15/47625] ✅ Saved: 01-25-2018.pdf
[16/47625] ✅ Saved: 01-26-2018.pdf
[17/47625] ✅ Saved: 01-27-2018.pdf
[18/47625] ✅ Saved: 01-28-2018.pdf
[19/47625] ✅ Saved: 01-29-2018.pdf
[20/47625] ✅ Saved: 01-30-2018.pdf
[21/47625] ✅ Saved: 01-31-2018.pdf
[22/47625] ✅ Saved: 01-01-2018.pdf
[23/47625] ✅ Saved: 01-02-2018.pdf
[24/47625] ✅ Saved: 01-03-2018.pdf
[25/47625] ✅ Saved: 01-04-2018.pdf
[26/47625] ✅ Saved: 01-05-2018.pdf
[27/47625] ✅ Saved: 01-06-2018.pdf
[28/47625] ✅ Saved: 01-07-2018.pdf
[29/47625] ✅ Saved: 01-08-201

unknown widths : 
[0, IndirectObject(42, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(46, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(50, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(54, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(58, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(62, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(66, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(70, 0, 5001588144)]
unknown widths : 
[0, IndirectObject(74, 0, 5001588144)]


[40150/47625] ✅ Saved: Library%20Literacy%20Coordinator%202021%20%281%29.pdf
[40151/47625] ✅ Saved: 3-22-21%20Housing%20Committee%20Meeting.pdf
[40154/47625] ✅ Saved: 3-22-21%20Personnel%20Committee%20Meeting.pdf


unknown widths : 
[0, IndirectObject(54, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(58, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(62, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(66, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(70, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(74, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(78, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(82, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(86, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(90, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(94, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(98, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(102, 0, 5001591136)]
unknown widths : 
[0, IndirectObject(106, 0, 5001591136)]


[40156/47625] ✅ Saved: REVISED-Backhoe%20Operator%20WS%202021_1.pdf
[40160/47625] ✅ Saved: 3-23-21%20Library%20Board%20of%20Trustees%20Meeting.pdf
[40161/47625] ✅ Saved: 3-23-21%20Ordinance%20Committee%20Meeting.pdf
[40162/47625] ✅ Saved: 3-25-21%20Merrimack%20Valley%20Workforce%20Board%20Meeting.pdf
[40163/47625] ✅ Saved: 4-7-21%20Planning%20Board%20Public%20Meeting-Hearing.pdf


...

[40787/47625] ✅ Saved: 6-28-21%20CANCELED-Personnel%20Committee%20Meeting.pdf
[40788/47625] ✅ Saved: 06-28-2021.pdf
[40793/47625] ✅ Saved: 6-29-21%20CANCELED-Personnel%20Committee%20Meeting.pdf
[40794/47625] ✅ Saved: 06-29-2021.pdf
[40795/47625] ✅ Saved: MEO-Laborer.pdf
[40796/47625] ✅ Saved: Motor%20Equipment%20Repairperson%20%28Diesel%20Mechanic%29.pdf
[40812/47625] ✅ Saved: 7-15-21%20Conservation%20Commission%20Meeting.pdf
[40813/47625] ✅ Saved: 7-15-21%20Conservation%20Commission%20Meeting.pdf
[40815/47625] ✅ Saved: Street%20Signs%20and%20Related%20Items%20Bid%20Results%2007.1.21%2011am%20Signed.pdf
[40826/47625] ✅ Saved: 2021.06%20Lawrence%20On-Call%20Bid%20Package.pdf
[40827/47625] ✅ Saved: 06-30-2021.pdf
[40828/47625] ✅ Saved: 07-02-2021.pdf
[40829/47625] ✅ Saved: 07-03-2021.pdf
[40830/47625] ✅ Saved: 07-01-2021.pdf
[40831/47625] ✅ Saved: 20210721%20-%20Re-Bid%20Student%20Enrichment%20Program%20Services%20%282021%20RFP%29.docx.pdf
[40832/47625] ✅ Saved: 07-05-2021.pdf
[40833/476

unknown widths : 
[0, IndirectObject(85, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(89, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(93, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(97, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(101, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(105, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(109, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(113, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(117, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(121, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(125, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(129, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(133, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(137, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(141, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(145, 0, 4998054144)]
unknown widths : 
[0, IndirectObject(149, 0, 4998054144)]
unknown widths : 


[40900/47625] ✅ Saved: 07142021%20Bid%20Results%20Island%20Street%20Pub%20In%20Imp%20Project%20IFB%202021%20Signed.pdf
[40901/47625] ✅ Saved: DOC071421-001.pdf
[40902/47625] ✅ Saved: 07-13-2021.pdf
[40903/47625] ✅ Saved: 07-14-2021.pdf
[40904/47625] ✅ Saved: 07-12-2021.pdf
[40905/47625] ✅ Saved: 7-21-21%20Merrimack%20Valley%20Planning%20Commission%20Public%20Hearings.pdf
[40906/47625] ✅ Saved: 8-4-21%20Planning%20Board%20Public%20Meeting-Hearing.pdf
[40907/47625] ✅ Saved: 8-4-21%20Planning%20Board%20Public%20Meeting-Hearing.pdf
[40910/47625] ✅ Saved: 7-20-21%20Oliver%20Partnership%20School%20Building%20Committee%20Meeting.pdf
[40933/47625] ✅ Saved: 7-21-21%20Bellevue%20Cemetery%20Board%20Meeting.pdf
[40934/47625] ✅ Saved: Hi-Special%20Heavy%20Motor%20Equipment%20Operator%202021.pdf
[40935/47625] ✅ Saved: Principle%20Accts%20Clerk%20-%20Elections%202021.pdf
[40936/47625] ✅ Saved: Principle%20Accts%20Clerk%20-%20City%20Clerk%202021.pdf
[40939/47625] ✅ Saved: GIC%2020%20percent%2021%20Pay

Multiple definitions in dictionary at byte 0xf9617 for key /PageMode


[41760/47625] ✅ Saved: Buckley%20Garage-Dwgs.%20A-4%20%20A-5%20Rev%201.pdf
[41761/47625] ✅ Saved: Frequently%20Asked%20Questions%20Regarding%20Reprecincting.pdf
[41762/47625] ✅ Saved: Information%20for%20Block%20Reports.pdf
[41763/47625] ✅ Saved: Information%20for%20Certified%20Vote%20of%20Adoption.pdf
[41764/47625] ✅ Saved: Information%20for%20Draft%20Map.pdf
[41765/47625] ✅ Saved: Lawrence_v1_092221.pdf
[41766/47625] ✅ Saved: Lawrence_v1_BR_092221.pdf
[41767/47625] ✅ Saved: Lawrence_v2_111221.pdf
[41768/47625] ✅ Saved: Reprecincting%20Checklist%20for%20Municipalities.pdf
[41771/47625] ✅ Saved: Sample%20Vote%20of%20Adoption%20Multi%20Precinct.pdf
[41772/47625] ✅ Saved: 11-17-21%20Conservation%20Commission%20Meeting.pdf
[41773/47625] ✅ Saved: 11-17-21%20Lawrence%20Redevelopment%20Authority%20Meeting.pdf
[41777/47625] ✅ Saved: Addendum%201%20Parking%20Management%20Services.pdf
[41782/47625] ✅ Saved: 11-11-2021.pdf
[41783/47625] ✅ Saved: 11-12-2021.pdf
[41784/47625] ✅ Saved: 11-13-2021.p

## Operation 2023, 2024

In [7]:
import os
import requests
import re
import calendar
from datetime import datetime

# Base data path using current notebook directory
notebook_dir = os.getcwd()
PDF_DATA_DIR = os.path.join(notebook_dir, 'data', 'pdfs')

# Date format: MM-DD-YYYY
DATE_PATTERN = re.compile(r"^(\d{2})-(\d{2})-(2018|2019|2020|2021|2022)\.pdf$", re.IGNORECASE)

def get_filename_from_content_disposition(header):
    if "filename=" in header.lower():
        return header.split("filename=")[-1].strip('"; ')
    return None

def parse_date_from_filename(filename):
    match = DATE_PATTERN.match(filename)
    if match:
        mm, dd, yyyy = int(match.group(1)), int(match.group(2)), int(match.group(3))
        return yyyy, mm, dd
    return None, None, None

def download_pdfs(start_id=45274, end_id=65274):
    os.makedirs(PDF_DATA_DIR, exist_ok=True)
    total = end_id - start_id + 1

    for count, doc_id in enumerate(range(start_id, end_id + 1), start=1):
        if count % 500 == 0:
            print(f"📦 Progress: {count} / {total} checked")

        url = f"https://lawpd.com/DocumentCenter/View/{doc_id}"
        try:
            res = requests.get(url, timeout=10)
            if res.status_code != 200 or "pdf" not in res.headers.get("Content-Type", "").lower():
                continue

            filename = get_filename_from_content_disposition(res.headers.get("Content-Disposition", ""))
            if not filename or not filename.lower().endswith(".pdf"):
                continue

            year, month, day = parse_date_from_filename(filename)
            if year is None:
                continue

            # Build folder path: .../data/pdfs/2023_january
            month_folder = f"{year}_{calendar.month_name[month].lower()}"
            save_dir = os.path.join(PDF_DATA_DIR, month_folder)
            os.makedirs(save_dir, exist_ok=True)

            save_path = os.path.join(save_dir, filename)
            with open(save_path, "wb") as f:
                f.write(res.content)

            print(f"[{count}/{total}] ✅ Saved: {filename}")

        except Exception:
            continue


In [8]:
download_pdfs()

[30/20001] ✅ Saved: 05-24-2022.pdf
[31/20001] ✅ Saved: 05-25-2022.pdf
[32/20001] ✅ Saved: 05-26-2022.pdf
[33/20001] ✅ Saved: 05-27-2022.pdf
[34/20001] ✅ Saved: 05-28-2022.pdf
[35/20001] ✅ Saved: 05-29-2022.pdf
[36/20001] ✅ Saved: 05-30-2022.pdf
[37/20001] ✅ Saved: 05-31-2022.pdf
[38/20001] ✅ Saved: 05-03-2022.pdf
[39/20001] ✅ Saved: 05-04-2022.pdf
[40/20001] ✅ Saved: 05-05-2022.pdf
[41/20001] ✅ Saved: 05-06-2022.pdf
[42/20001] ✅ Saved: 05-07-2022.pdf
[43/20001] ✅ Saved: 05-08-2022.pdf
[44/20001] ✅ Saved: 05-09-2022.pdf
[45/20001] ✅ Saved: 05-10-2022.pdf
[46/20001] ✅ Saved: 05-11-2022.pdf
[47/20001] ✅ Saved: 05-12-2022.pdf
[48/20001] ✅ Saved: 05-13-2022.pdf
[49/20001] ✅ Saved: 05-14-2022.pdf
[50/20001] ✅ Saved: 05-15-2022.pdf
[51/20001] ✅ Saved: 05-16-2022.pdf
[52/20001] ✅ Saved: 05-17-2022.pdf
[53/20001] ✅ Saved: 05-18-2022.pdf
[54/20001] ✅ Saved: 05-19-2022.pdf
[55/20001] ✅ Saved: 05-20-2022.pdf
[56/20001] ✅ Saved: 05-21-2022.pdf
[57/20001] ✅ Saved: 05-22-2022.pdf
[58/20001] ✅ Saved: 